In [1]:
import requests
import csv
import subprocess
import json
from tqdm import tqdm
import time

In [2]:
M3U_FILE = "../tung_iptv.m3u"
OUTPUT_FILE = "channel_status.csv"

HEADERS = {
    "User-Agent": "TiviMate/5.2.0 (Linux; Android 10)"
}

def parse_m3u(file_path):
    channels = []
    name = None
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#EXTINF"):
                name = line.split(",")[-1]
            elif line.startswith("http"):
                channels.append({
                    "name": name,
                    "url": line
                })
    return channels


def check_stream(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=2, stream=True)
        if r.status_code == 200:
            return "ONLINE"
        else:
            return f"HTTP {r.status_code}"
    except Exception:
        return "OFFLINE"
    
def check_stream_performance(url):
    start_time = time.perf_counter()
    try:
        r = requests.get(url, headers=HEADERS, timeout=2, stream=True)
        end_time = time.perf_counter()
        if r.status_code == 200:
            return f"{(end_time - start_time) * 1000:.2f}ms"
    except Exception:
        return "OFFLINE"    

In [3]:
channels = parse_m3u(M3U_FILE)
rows = []
for ch in tqdm(channels, desc="Checking channels"):
    rows.append([
        ch["name"],
        ch["url"],
        check_stream(ch["url"]),
        check_stream_performance(ch["url"])
    ])

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Channel Name", "URL", "Status", "Response"])
    writer.writerows(rows)

print("Done. Results saved to:", OUTPUT_FILE)

Checking channels: 100%|██████████| 148/148 [08:34<00:00,  3.47s/it]

Done. Results saved to: channel_status.csv
